# 🏦 Rule-Based Synthetic Data with Business Constraints

In this notebook we will build a rule-based synthetic data script that enforces key business constraints such as valid ranges and consistent categorical values, then validate each rule.

We have the following scenario: 

You've built a banking dashboard application and need realistic transactional data to test it — but you have no real customer data available.
1. Define the data requirements as a structured checklist
2. Translate the requirements into a schema
3. Generate 1,000 synthetic banking transactions
4. Validate the generated dataset

## 1. Define Requirements

Before writing a single line of generation code, we answer four questions:

| # | Question | Answer (our banking transactions) |
|---|---|---|
| ✅ | **What columns does the table need?** | `transaction_id`, `account_id`, `transaction_date`, `transaction_type`, `merchant`, `amount`, `country` |
| ✅ | **What are the valid ranges?** | Amounts: €1 – €5,000 · Dates: 2023-01-01 to 2024-12-31 |
| ✅ | **What business constraints must hold?** | Country code must be `DE` (Germany) · Amount > 0 and ≤ €5,000 · No null values · `transaction_id` is unique |
| ✅ | **What behavioural patterns should the data reflect?** | 60% debits / 40% credits |

These four questions become the blueprint for our schema.

## 2. Define the Schema

We translate each requirement directly into a schema dictionary.
Each key is a column name; the value describes its type, format, and constraints.

In [7]:
from datetime import datetime

schema = {
    # --- Identifiers ---
    "transaction_id": {
        "datatype":  "string",
        "vartype":   "string",
        "regex":     "TXN-[0-9]{8}",          # e.g. TXN-00482931
    },
    "account_id": {
        "datatype":  "string",
        "vartype":   "string",
        "regex":     "ACC-[0-9]{6}",           # e.g. ACC-004829
    },

    # --- When ---
    "transaction_date": {
        "datatype":  "date",
        "vartype":   "date",
        "min":       datetime(2023, 1, 1),
        "max":       datetime(2024, 12, 31),
        "format":    "%Y-%m-%d",
    },

    # --- What kind ---
    "transaction_type": {
        "datatype":    "categorical",
        "vartype":     "string",
        "categories": {"debit": 60, "credit": 40},  # 60% debits, 40% credits
    },

    # --- Where ---
    "merchant": {
        "datatype":  "string",
        "vartype":   "string",
        "characteristic": "name",               # realistic merchant names via Faker
    },

    # --- How much ---
    "amount": {
        "datatype":  "numerical",
        "vartype":   "float",
        "min":       1,
        "max":       5_000,                     # transaction amounts: €1 – €5,000
    },

    # --- Country ---
    "country_code": {
        "datatype":    "categorical",
        "vartype":     "string",
        "categories": {"DE": 40, "US": 30, "GB": 20, "FR": 10},             # Germany only (ISO 3166-1 alpha-2)
    },
}

print(f"Schema defined: {len(schema)} columns ✓")
for col, cfg in schema.items():
    print(f"  • {col:<20} {cfg['datatype']}")

Schema defined: 7 columns ✓
  • transaction_id       string
  • account_id           string
  • transaction_date     date
  • transaction_type     categorical
  • merchant             string
  • amount               numerical
  • country_code         categorical


## 3. Build Metadata & Initialise the Synthesizer

We pass the schema dictionary to `MetadataConfigurationBuilder`, which creates a `Metadata` object — a reusable blueprint that describes the dataset structure. We can save it and regenerate identical-structure datasets at any time.

In [8]:
from ydata.metadata import Metadata
from ydata.metadata.builder import MetadataConfigurationBuilder
from ydata.synthesizers import FakerSynthesizer

# Build metadata from the schema dictionary
builder = MetadataConfigurationBuilder(schema)
metadata = Metadata(configuration_builder=builder)

print("Metadata object created ✓")
print(metadata)

Metadata object created ✓
Metadata Summary 
 
Dataset type: TABULAR
Dataset attributes: 
Number of columns: 7
Number of rows: 0
Duplicate rows: 0
Target column: 

Column detail: 
             Column    Data type Variable type Characteristics
0    transaction_id       string        string                
1        account_id       string        string                
2  transaction_date         date          date                
3  transaction_type  categorical        string                
4          merchant       string        string            name
5            amount    numerical         float                
6      country_code  categorical        string                
Warnings: 
No warning!



## 4. Fit & Generate

The `FakerSynthesizer` uses the `locale` parameter to ensure names, dates, and other locale-sensitive fields are generated in the right language and format. Here we use English (`"en"`).

We `fit` on the metadata object, then `sample` as many rows as we need.

In [9]:
# Instantiate with English locale
synth = FakerSynthesizer(locale="en")

# Fit on the metadata (learns the schema structure)
synth.fit(metadata)

# Generate 1,000 transactions
# You can request as many rows as you need
sample = synth.sample(sample_size=1000)
df = sample.to_pandas()

print(f"Generated: {df.shape[0]:,} rows × {df.shape[1]} columns ✓")
df.head()

Generated: 1,000 rows × 7 columns ✓


,transaction_id,account_id,transaction_date,transaction_type,merchant,amount,country_code
0,TXN-73484424,ACC-115512,2023-01-22,credit,Neil Campbell,910.279113,DE
1,TXN-66839341,ACC-744264,2024-02-18,debit,Wesley Smith,4533.000983,US
2,TXN-34282076,ACC-091042,2023-01-31,debit,Sarah Murphy,392.377814,DE
3,TXN-56898577,ACC-752613,2024-08-20,debit,Jocelyn Choi,1486.053127,FR
4,TXN-68476285,ACC-783467,2024-04-13,debit,Pamela Murphy DDS,3986.391475,US


In [10]:
# Derive the country name from the country code
CODE_TO_COUNTRY = {
    "DE": "Germany",
    "US": "USA",
    "GB": "UK",
    "FR": "France",
}

df["country"] = df["country_code"].map(CODE_TO_COUNTRY)

print("Country name column derived ✓")
df[["country_code", "country"]].value_counts().sort_index()

Country name column derived ✓


country_code  country
DE            Germany    400
FR            France     100
GB            UK         200
US            USA        300
Name: count, dtype: int64

## 5. Define Constraints with the Constraint Engine

The schema already instructs the synthesizer to stay within valid ranges, but we want to **formally declare each business rule as a constraint object** — so they are explicit, reusable, and machine-checkable.

We use ydata-sdk's ConstraintEngine for this, though the same rules could equally be expressed with Great Expectations or even as plain Python functions. Each constraint maps to one business rule:

| Constraint | Rule |
|---|---|
| `NotNull` | No column may contain missing values |
| `Unique` | Every `transaction_id` must be unique |
| `GreaterThan(0)` | Every amount must be strictly positive |
| `MaxBetween(0, 5000)` | No amount may exceed €5,000 |
| `CustomConstraint` | Country code must be mapped to the column Country

`GreaterThan` and `CustomConstraint` are **row-level**: they tag non-compliant rows and can filter them out.
`MaxBetween` is a **column-level** summary check: it verifies the column's maximum falls within a bound.

In [12]:
from ydata.constraints import (
    ConstraintEngine,
    NotNull,
    Unique,
    GreaterThan,
    MaxBetween,
    CustomConstraint,
)
from ydata.dataset import Dataset

# Define one constraint per business rule
engine = ConstraintEngine([
    # No missing values anywhere in the dataset
    NotNull(
        columns=list(df.columns),
        name="no_missing_values",
    ),

    # transaction_id must be unique across all rows
    Unique(
        columns=["transaction_id"],
        name="unique_transaction_id",
    ),

    # Amounts must be strictly positive  (row-level: flags violating rows)
    GreaterThan(
        columns=["amount"],
        value=0,
    ),

    # Amount ceiling: column max must stay within €0–€5,000  (column-level: summary check)
    MaxBetween(
        lower_bound=0,
        upper_bound=5_000,
        columns=["amount"],
        name="amount_ceiling",
    ),

    # All transactions must be in Germany (row-level: flags non-DE rows)
    CustomConstraint(
    lambda row: CODE_TO_COUNTRY.get(row["country_code"]) == row["country"],
    columns=["country_code", "country"],
    name="country_code_consistency",
    axis="row",
    ),
])

# Print a human-readable description of every declared rule
engine.explain_constraints()
print("Constraint engine ready ✓")

Constraint engine ready ✓


## 6. Validate with the Constraint Engine

`engine.validate(dataset)` runs every declared rule against the data and records which rows pass or fail each one. `engine.summary()` aggregates the results into a table — one row per constraint, showing pass rate and violation count.

In [13]:
# Wrap the DataFrame in a Dataset so the engine can process it
dataset = Dataset(df)

# Run all constraints
engine.validate(dataset)

# Show a summary table: one row per constraint
engine.summary()

/Users/fabianaclemente/Documents/github/ydata/src/ydatautils/ydata-utils/ydata/dataset/dataset.py:62: UserWarning: All the input Variable Types were set as `string` or `object`. It is recommend to revise the VariableType settings for optimal results.
  warnings.warn(


{'rows_violation_count': np.int64(0),
 'rows_violation_ratio': 0.0,
 'violation_per_constraint': {'no_missing_values': {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0},
  'unique_transaction_id': {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0},
  "GreaterThan(columns=['amount'], value=0)": {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0},
  'amount_ceiling': {'column_violation_count': np.int64(0),
   'column_violation_ratio': 0.0,
   'violated_columns': [],
   'validation_time': 0.0},
  'country_code_consistency': {'rows_violation_count': np.int64(0),
   'rows_violation_ratio': 0.0,
   'validation_time': 0.0}}}

In [14]:
# Filter to only rows that pass every row-level constraint
clean_df = engine.filter(dataset).to_pandas()

print(f"Rows before filtering: {len(df):,}")
print(f"Rows after filtering:  {len(clean_df):,}  (all pass every declared rule)")
clean_df.head(10)

Rows before filtering: 1,000
Rows after filtering:  1,000  (all pass every declared rule)


/var/folders/rg/7f49h9cd5hsfq8yj7rtxlkpm0000gn/T/ipykernel_63702/3887236786.py:2: UserWarning: filter() operates on row constraints only. The following column constraint(s) will be ignored during filtering: ['amount_ceiling']. Use validate() + summary() to inspect column-level violations.
  clean_df = engine.filter(dataset).to_pandas()
/Users/fabianaclemente/Documents/github/ydata/src/ydatautils/ydata-utils/ydata/dataset/dataset.py:62: UserWarning: All the input Variable Types were set as `string` or `object`. It is recommend to revise the VariableType settings for optimal results.
  warnings.warn(


,transaction_id,account_id,transaction_date,transaction_type,merchant,amount,country_code,country
0,TXN-73484424,ACC-115512,2023-01-22,credit,Neil Campbell,910.279113,DE,Germany
1,TXN-66839341,ACC-744264,2024-02-18,debit,Wesley Smith,4533.000983,US,USA
2,TXN-34282076,ACC-091042,2023-01-31,debit,Sarah Murphy,392.377814,DE,Germany
3,TXN-56898577,ACC-752613,2024-08-20,debit,Jocelyn Choi,1486.053127,FR,France
4,TXN-68476285,ACC-783467,2024-04-13,debit,Pamela Murphy DDS,3986.391475,US,USA
5,TXN-44854233,ACC-668641,2024-03-09,credit,Michael Ingram,4424.565063,FR,France
6,TXN-61335569,ACC-604909,2023-06-12,credit,Sheryl Johnson,3425.531233,DE,Germany
7,TXN-58996895,ACC-917449,2023-07-12,debit,Heather Orr,4318.178641,DE,Germany
8,TXN-11013266,ACC-391759,2024-04-19,debit,David White,2001.108509,FR,France
9,TXN-25510342,ACC-243944,2023-08-13,credit,Sheena Jackson,3081.307040,US,USA


In [15]:
# --- Export the validated dataset ---
output_path = "banking_transactions_synthetic.csv"
clean_df.to_csv(output_path, index=False)
print(f"Dataset saved → {output_path} ({len(clean_df):,} rows) ✓")
clean_df.head()

Dataset saved → banking_transactions_synthetic.csv (1,000 rows) ✓


,transaction_id,account_id,transaction_date,transaction_type,merchant,amount,country_code,country
0,TXN-73484424,ACC-115512,2023-01-22,credit,Neil Campbell,910.279113,DE,Germany
1,TXN-66839341,ACC-744264,2024-02-18,debit,Wesley Smith,4533.000983,US,USA
2,TXN-34282076,ACC-091042,2023-01-31,debit,Sarah Murphy,392.377814,DE,Germany
3,TXN-56898577,ACC-752613,2024-08-20,debit,Jocelyn Choi,1486.053127,FR,France
4,TXN-68476285,ACC-783467,2024-04-13,debit,Pamela Murphy DDS,3986.391475,US,USA


## ✅ Summary

In this notebook you:

1. **Defined requirements** as a structured four-question checklist before writing any code
2. **Translated them into a schema** dictionary covering identifiers, dates, categories, and numeric ranges.
3. **Generated 1,000 synthetic banking transactions** with `FakerSynthesizer`
5. **Declared business rules as constraint objects** for ease of validation and re-use.
6. **Validated the generated datasets**

> **Active Learning:** Want to describe a dataset in plain language instead of writing a schema by hand? Check the companion exercise: *Text-to-Dataset with LLMSynthesizer*.